# AeroFlow Exploratory Analysis

This notebook analyses AeroFlow's supply chain performance using the prepared datasets created during the data cleaning stage.

The analysis focuses on delivery reliability, supplier performance, inventory risk, demand and forecasting, and quality performance to identify operational risks and opportunities for improvement.

## 1. Import Libraries and Load Clean Data

In [3]:
import pandas as pd

parts = pd.read_csv("../data/clean/parts_master_clean.csv")
purchase_orders = pd.read_csv("../data/clean/purchase_orders_clean.csv")
quality_incidents = pd.read_csv("../data/clean/quality_incidents_clean.csv")
supply_chain_history = pd.read_csv("../data/clean/supply_chain_history_clean.csv")

In [4]:
parts.shape, purchase_orders.shape, quality_incidents.shape, supply_chain_history.shape



((300, 9), (29666, 14), (368, 8), (280800, 13))

## 2. Overall Supply Chain Performance
### OTIF Performance


In [5]:
# Calculate overall OTIF rate
otif_pct = purchase_orders['otif_flag'].mean() * 100
round(otif_pct, 2)

np.float64(39.74)

AeroFlow achieved an overall OTIF rate of 39.74%, meaning fewer than half of purchase orders were delivered both on time and in full. This indicates a significant opportunity to improve supplier delivery reliability.

In [ ]:
# Calculate on-time rate
on_time_pct = purchase_orders['on_time_flag'].mean() * 100
round(on_time_pct, 2)


np.float64(44.15)

AeroFlow achieved an overall on-time delivery rate of 44.15%. This is slightly higher than the OTIF rate of 39.74%, indicating that some orders arrived on time but did not meet the full quantity requirement.

In [11]:
# Calculate in-full rate
in_full_pct = purchase_orders['in_full_flag'].mean() * 100
round(in_full_pct, 2)


np.float64(88.69)

AeroFlow achieved an in-full delivery rate of 88.69%, which is substantially stronger than its on-time rate of 44.15%. This suggests delivery delays are the main factor reducing overall OTIF performance.

In [13]:
# Calculate average delivery variance
delivery_variance_days = purchase_orders["delivery_variance_days"].mean()
round(delivery_variance_days, 2)                                                                                                                                                                                  

np.float64(1.18)

AeroFlow deliveries arrived an average of 1.18 days later than the promised delivery date. Combined with the low on-time rate of 44.15%, this confirms that delivery timeliness is a key operational weakness.


In [18]:
# Calculate stockout rate
stockout_rate = supply_chain_history['stockout_flag'].mean() * 100
round(stockout_rate, 2)


np.float64(0.57)

AeroFlow recorded a stockout rate of 0.57%, meaning stockouts occurred in a small proportion of weekly part-site records. Although the overall rate is low, later analysis should identify whether stockouts are concentrated among critical parts, specific sites or particular suppliers.

In [19]:
# Calculate backorder rate
backorder_rate = supply_chain_history['backorder_flag'].mean() * 100
round(backorder_rate, 2)

np.float64(0.83)

AeroFlow recorded a backorder rate of 0.83%. Although this is low overall, the next stage should examine whether backorders are concentrated among specific parts, sites or suppliers.

## 3. Supplier Performance

In [74]:
supplier_summary = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        order_count=("po_id", "count"),
        on_time_rate=("on_time_flag", "mean"),
        in_full_rate=("in_full_flag", "mean"),
        otif_rate=("otif_flag", "mean"),
        avg_delivery_delay_days=("delivery_variance_days", "mean"),
    )
    .reset_index()
)

supplier_summary.head(10)

,supplier_id,order_count,on_time_rate,in_full_rate,otif_rate,avg_delivery_delay_days
0,SUP001,659,0.578149,0.918058,0.520486,0.267071
1,SUP002,763,0.281782,0.871560,0.256881,2.245085
2,SUP003,633,0.330174,0.845182,0.281201,1.914692
3,SUP004,662,0.564955,0.922961,0.525680,0.256798
4,SUP005,436,0.337156,0.848624,0.282110,1.931193
5,SUP006,1096,0.313869,0.849453,0.263686,2.114051
6,SUP007,831,0.531889,0.925391,0.489771,0.393502
7,SUP008,587,0.560477,0.916525,0.514480,0.183986
8,SUP009,649,0.302003,0.852080,0.252696,1.927581
9,SUP010,930,0.567742,0.917204,0.519355,0.352688


SUP033 is the weakest performing supplier, with an OTIF rate of 3.01% across 631 purchase orders. Its on-time rate is only 3.65%, while its in-full rate is 76.23%, indicating that chronic delivery lateness is the primary driver of poor OTIF performance.

In [27]:
parts.loc[
    parts["supplier_id_primary"] == "SUP033", ["part_id", "criticality_class", "supplier_risk_class"]]


,part_id,criticality_class,supplier_risk_class
60,P00061,C,High
126,P00127,C,High
128,P00129,C,High
163,P00164,C,High
255,P00256,B,High
263,P00264,A,High
293,P00294,B,High


SUP033 primarily supplies seven AeroFlow parts, all of which are classified as High supplier risk. The portfolio includes one Class A and two Class B criticality parts, increasing the operational significance of SUP033's poor delivery performance.

In [30]:
sup033_avg_delivery_variance = (
purchase_orders.loc[
    purchase_orders["supplier_id"] == "SUP033",
    "delivery_variance_days"
]
    .mean()
)

round(sup033_avg_delivery_variance, 2)

np.float64(6.09)

SUP033 deliveries arrived an average of 6.09 days later than promised, compared with AeroFlow's overall average delay of 1.18 days. This reinforces that SUP033's poor OTIF performance is primarily driven by persistent and significant delivery lateness.

In [34]:
supplier_risk_summary = purchase_orders.merge(
    parts,
    on="part_id",
    how="left"
)
supplier_risk_summary.shape

supplier_risk_summary["supplier_risk_class"].value_counts()

supplier_risk_class
Low       15408
Medium    13627
High        631
Name: count, dtype: int64

In [35]:
risk_otif = (
    supplier_risk_summary
    .groupby("supplier_risk_class")["otif_flag"]
    .mean()
    * 100
)

risk_otif

supplier_risk_class
High       3.011094
Low       52.518172
Medium    26.990533
Name: otif_flag, dtype: float64

OTIF performance declines sharply as supplier risk increases. Low-risk suppliers achieved 52.52% OTIF, compared with 27.00% for medium-risk suppliers and 3.01% for the high-risk category. However, the high-risk result is entirely driven by SUP033, so it should be treated as evidence of a major supplier specific risk rather than a broad conclusion about all high-risk suppliers.

In [39]:
supplier_avg_delivery_delay = (
    purchase_orders
    .groupby("supplier_id")["delivery_variance_days"]
    .mean().sort_values(ascending=False)
.reset_index())
supplier_avg_delivery_delay.head(10)

,supplier_id,delivery_variance_days
0,SUP033,6.088748
1,SUP023,2.367857
2,SUP002,2.245085
3,SUP015,2.205811
4,SUP039,2.179739
5,SUP026,2.158824
6,SUP006,2.114051
7,SUP014,2.081851
8,SUP021,2.078394
9,SUP025,2.068140


SUP033 recorded the highest average delivery delay at 6.09 days late, more than double the next-worst supplier. This confirms that SUP033 is a major delivery reliability outlier and should be treated as a priority supplier risk.

In [78]:
quality_performance = (
    quality_incidents
    .groupby("supplier_id")
    .agg(incident_count=("incident_id", "count"))
    .reset_index()
    .sort_values("incident_count", ascending=False)
)

quality_performance.head(10)

quality_performance[
    quality_performance["supplier_id"] == "SUP033"
]

,supplier_id,incident_count
32,SUP033,11


SUP033 recorded 11 quality incidents, which is not among the highest supplier incident counts. This suggests that its primary performance issue is delivery reliability rather than quality frequency.

In [79]:
supplier_quality_rate = supplier_order_count_clean.merge(
    quality_performance,
    on="supplier_id",
    how="left"
)

supplier_quality_rate["incident_count"] = (
    supplier_quality_rate["incident_count"].fillna(0)
)

supplier_quality_rate.head()

,supplier_id,order_count,incident_count
0,SUP001,659,3
1,SUP002,763,10
2,SUP003,633,6
3,SUP004,662,6
4,SUP005,436,1


In [80]:
supplier_quality_rate["quality_incidents_per_100_orders"] = (
    supplier_quality_rate["incident_count"]
    / supplier_quality_rate["order_count"]
    * 100
)

supplier_quality_rate = (
    supplier_quality_rate
    .sort_values(
        "quality_incidents_per_100_orders",
        ascending=False
    )
    .reset_index(drop=True)
)

supplier_quality_rate.head(10)


,supplier_id,order_count,incident_count,quality_incidents_per_100_orders
0,SUP023,280,7,2.500000
1,SUP024,821,16,1.948843
2,SUP027,1311,24,1.830664
3,SUP018,607,11,1.812191
4,SUP040,1215,22,1.810700
5,SUP033,631,11,1.743265
6,SUP034,578,10,1.730104
7,SUP009,649,11,1.694915
8,SUP013,1156,19,1.643599
9,SUP006,1096,18,1.642336


When adjusted for purchase-order volume, SUP033 recorded 1.74 quality incidents per 100 orders, ranking sixth highest among suppliers. This indicates some quality exposure, but delivery reliability remains the more severe issue for SUP033.

In [55]:
quality_incidents["defect_severity"].value_counts()



defect_severity
Minor       243
Major        91
Critical     34
Name: count, dtype: int64

In [81]:
serious_incidents = quality_incidents[
    quality_incidents["defect_severity"].isin(["Major", "Critical"])
]


SUP033 recorded seven Major or Critical quality incidents, placing it among the suppliers with the highest number of serious quality issues. Although its total incident count was not the highest, the severity profile adds further concern alongside its poor delivery performance.

In [71]:
supplier_order_count_clean = (
    purchase_orders
    .groupby("supplier_id")
    .agg(order_count=("po_id", "count"))
    .reset_index()
)

serious_incident_count = (
    serious_incidents
    .groupby("supplier_id")
    .agg(serious_incident_count=("incident_id", "count"))
    .reset_index()
)

serious_incident_rate = supplier_order_count_clean.merge(
    serious_incident_count,
    on="supplier_id",
    how="left"
)

serious_incident_rate.head()

,supplier_id,order_count,serious_incident_count
0,SUP001,659,1.0
1,SUP002,763,2.0
2,SUP003,633,4.0
3,SUP004,662,NaN
4,SUP005,436,NaN


In [73]:
serious_incident_rate["serious_incident_count"] = (
    serious_incident_rate["serious_incident_count"].fillna(0)
)

serious_incident_rate["serious_incidents_per_100_orders"] = (
    serious_incident_rate["serious_incident_count"]
    / serious_incident_rate["order_count"]
    * 100
)

serious_incident_rate = (
    serious_incident_rate
    .sort_values(
        "serious_incidents_per_100_orders",
        ascending=False
    )
    .reset_index(drop=True)
)

serious_incident_rate.head(10)

,supplier_id,order_count,serious_incident_count,serious_incidents_per_100_orders
0,SUP033,631,7.0,1.109350
1,SUP016,385,4.0,1.038961
2,SUP025,543,5.0,0.920810
3,SUP034,578,5.0,0.865052
4,SUP030,1171,9.0,0.768574
5,SUP038,967,7.0,0.723888
6,SUP019,691,5.0,0.723589
7,SUP037,842,6.0,0.712589
8,SUP027,1311,9.0,0.686499
9,SUP008,587,4.0,0.681431


When adjusted for purchase-order volume, SUP033 recorded the highest rate of Major or Critical quality incidents at 1.11 per 100 orders. Combined with its 3.01% OTIF rate, 6.09-day average delivery delay and High supplier-risk classification, SUP033 represents AeroFlow's most significant supplier performance risk.

## 4. Inventory and Parts Risk